# Hello world

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from collections import Counter
import time
import os

# Train/val split

* Split the data with 20% of the data as validation
* Batch size standard 32
* Img size as 224,224 since imagenet trained on 224*224
* Seed 1337 for reproducibility


In [ ]:
Data_DIR = "../data/images/"
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 1337
Epochs = 2

train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    Data_DIR,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",

)

val_ds_raw = tf.keras.utils.image_dataset_from_directory(
    Data_DIR,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",

)

class_names = train_ds_raw.class_names
num_classes = len(class_names)


train_ds = train_ds_raw
val_ds = val_ds_raw


AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(AUTOTUNE)
val_ds = val_ds.cache().prefetch(AUTOTUNE)


In [ ]:
print(IMG_SIZE + (3,), BATCH_SIZE, SEED)


## Corrupt

* Corrupt diffrent amount of the training data to se how sensitive the model is.
* Evaluate the model on corrupt data to se how robust the model is.


### Random noice

* Change random pixels from the standard 255 colors by changing the values by the mean
* a negative *mean* value will give darker pixels since 0,0,0 is black and a positive mean will skew the imges to brighter pictures.

In [ ]:
def add_noise(img, label, std=255.0,mean=-50.0):
    noise = tf.random.normal(tf.shape(img), mean=mean, stddev=std)
    img = tf.clip_by_value(img + noise, 0.0, 255.0)
    return img, label

val_ds_noise = (val_ds_raw
                .map(lambda img, label: add_noise(img, label, std=40.0)))

### Jpeg compression

* cast to convert to 0-255
* convert to x using a encode_jpeg, now a compressed byte string from img with a quality from 0-100.
* decode converts back to img but with quality loss with rgd channels.
* cast makes the image a tensor again.

In [ ]:
def jpeg_compression(img, label, quality=1):
    img = tf.cast(img, tf.uint8)                 
    x = tf.io.encode_jpeg(img, quality=quality)
    img = tf.io.decode_jpeg(x, channels=3)
    img = tf.cast(img, tf.float32)
    return img, label


val_ds_jpeg = (val_ds_raw
               .unbatch()
               .map(lambda x, y: jpeg_compression(x, y),)
               .batch(BATCH_SIZE))

### Blur

* expand_dims to create correct format for avg_pool2d = 1(picture from batch),224(height),224(witdth),3(colors)
* ksize=k how many pixels should be avaraged out, k*k mean, stride=how many pixels to move each time
* squeeze to remove first dimension

In [ ]:
def blur(img, label, k=5):
    img4 = tf.expand_dims(img, 0) 
    img4 = tf.nn.avg_pool2d(img4, ksize=k, strides=1, padding="SAME")
    img = tf.squeeze(img4, 0)      
    return img, label

val_ds_blur = (val_ds_raw
               .unbatch()
               .map(lambda x, y: blur(x, y),)
               .batch(BATCH_SIZE))


## Corrupt some input data

* corrupt part of the data based on p
* unbatch and give each image , a index with enumerate
* i = index , data = img and label
* use random.stateless_uniform to create same random nr each time based on indecis.
* corrupt if r < p
* map to all apply chance to all images

### mixed corruption

* img index i as seed
* corrupts a 1/3 of data to blur,noice and jpeg each based on stateless generator


In [ ]:
def corrupt_fraction(train_ds_raw, corrupt_fn, p=0.5, seed=SEED, batch_size=32, shuffle=True):

    ds = train_ds_raw.unbatch().enumerate() # Add an index to each element

    def maybe_corrupt(i, data): 
        img, label = data 

        # Create a random number in [0, 1) using a stateless random generator
        r = tf.random.stateless_uniform(
            shape=[], 
            seed=tf.stack([tf.cast(seed, tf.int32), tf.cast(i, tf.int32)]), # Use the index as part of the seed to ensure different random numbers for each element 
        )
    
        return tf.cond(
            r < p,
            lambda: corrupt_fn(i, img, label), # true case: apply corruption
            lambda: (img, label) # false case: return original
        )

    ds = ds.map(maybe_corrupt) 
    # if shuffle:
    #     ds = ds.shuffle(2000, seed=seed, reshuffle_each_iteration=True)

    ds = ds.batch(batch_size)
    return ds

In [ ]:
def mixed_corruption(i, img, label):
    i = tf.cast(i, tf.int32)

    r_type = tf.random.stateless_uniform(
        shape=[],
        seed=tf.stack([tf.constant(SEED + 1, tf.int32), i])
    )

    return tf.case(
        [
            (r_type < 1/3, lambda: blur(img, label, k=5)),
            (r_type < 2/3, lambda: add_noise(img, label, std=25.0)),
        ],
        default=lambda: jpeg_compression(img, label, quality=30),
        exclusive=False
    )

# Build model


EfficentNetB0 is based on images from the imagenet database containing 1000 categories including dogbreeds some over 1m pictures.
* ```include_top=False``` to ignore efficientnet classification, but we keep the feature extraction
* ```weights="imagenet"``` keep their pretrained weights for those features.
* 224 * 224 * 3, w,b,colors
* ```base.trainable = False``` to keep only using imagenet features
* ```inputs``` get w,b,c input > ```preprocess_input``` scales to correct format > ```x = base(x, training=False)``` feature representation from imagenet
* ``GlobalAveragePooling2D`` takes avarage and makes it 1d vector
* ``outputs`` creates a value from 0-1 by using *softmax* for each of the 120 classes

* use adam as optimizer with a learning rate of 0.001
* ``SparseCategoricalCrossentropy`` CategoricalCrossentropy since we have a *int* for our dog classses, then predicts a probability for all dog classes.



In [ ]:
def build_model():
    base = tf.keras.applications.EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_shape=IMG_SIZE + (3,)
    )
    base.trainable = False

    inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
    x = tf.keras.applications.efficientnet.preprocess_input(inputs)
    x = base(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

    model = tf.keras.Model(inputs, outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=[
            tf.keras.metrics.SparseCategoricalAccuracy(name="accuracy"),
            tf.keras.metrics.SparseTopKCategoricalAccuracy(k=5, name="top5"),
        ],
    )
    return model

## logg values

* callback for tensorboard

### predictions

* Save predictions and probability

In [ ]:
def make_callbacks(corruption_name, p):
    run_name = f"{corruption_name}_p{int(p*100)}_{datetime.now().strftime('%Y%m%d-%H%M%S')}"
    log_dir = os.path.join("logs", run_name)

    tb = tf.keras.callbacks.TensorBoard(
        log_dir=log_dir,
        histogram_freq=0,
        write_graph=True,
        update_freq="batch",
        profile_batch=0
    )

    return [tb], log_dir

In [ ]:
def collect_predictions(model, ds):
    y_true = []
    y_pred = []
    y_prob = []

    for images, labels in ds:
        probs = model.predict(images, verbose=0)
        preds = np.argmax(probs, axis=1)

        y_true.extend(labels.numpy())
        y_pred.extend(preds)
        y_prob.extend(probs)

    return np.array(y_true), np.array(y_pred), np.array(y_prob)

## Build models

* Build diffrent models for diffrent corrupt values
* take traindata and run corrupt fraction to get a % of corrupt pictures
* If the picture is choosen to be corrupt corrupt_fn=mixed_corruption chooses the type of corruption. 

In [ ]:
corruption_levels = [1]

results = []
predictions_store = []

for p in corruption_levels:
    print(f"Training with {(p*100)}% corrupted images with")

    train_variant = corrupt_fraction(
        train_ds_raw,
        corrupt_fn=mixed_corruption,
        p=p,
        seed=SEED,
        batch_size=BATCH_SIZE,
    )

    model = build_model()
    callbacks, log_dir = make_callbacks("mixed", p)

    start = time.perf_counter()

    history = model.fit(
        train_variant,
        validation_data=val_ds,
        epochs=Epochs,
        callbacks=callbacks,
    )

    train_time = time.perf_counter() - start

    val_loss, val_top1, val_top5 = model.evaluate(val_ds, verbose=0)   
    y_true, y_pred, y_prob = collect_predictions(model, val_ds)

    predictions_store.append({
        "corruption_type": "mixed",
        "p": p,
        "y_true": y_true,
        "y_pred": y_pred,
        "y_prob": y_prob,
    })

    results.append({
        "corruption_type": "mixed",
        "p": p,
        "history" : history.history,
        "val_loss": val_loss,
        "val_top1": val_top1,
        "val_top5": val_top5,
        "train_time": train_time,
        "log_dir": log_dir
    })

# Results

* loops trough stored predictions for p
* r object becomes run with all preditions
* loop trough class index with corresponding dog
* mask how many it got right
*    

In [ ]:
def class_accuracy_report(predictions_store, p, class_names, top_n=5):
    run = None
    for r in predictions_store:
        if r["p"] == p:
            run = r
            break

    y_true = run["y_true"]
    y_pred = run["y_pred"]

    per_class = []

    for class_idx, class_name in enumerate(class_names):
        mask = (y_true == class_idx)
        total = np.sum(mask)

        correct = np.sum(y_pred[mask] == y_true[mask]) # compare how many are true vs predicted for this class
        acc = correct / total

        per_class.append({
            "class_idx": class_idx,
            "class_name": class_name.split("-")[1],
            "total": total,
            "correct": correct,
            "accuracy": acc,
        })

    per_class = sorted(per_class, key=lambda x: x["accuracy"], reverse=True)

    print("\nMest rätt på:")
    for row in per_class[:top_n]:
        print(f'{row["class_name"]:30s} acc={row["accuracy"]:.3f} ({row["correct"]}/{row["total"]})')

    print("\nMest fel på:")
    for row in per_class[-top_n:]:
        print(f'{row["class_name"]:30s} acc={row["accuracy"]:.3f} ({row["correct"]}/{row["total"]})')

In [ ]:
class_accuracy_report(predictions_store, 1, class_names)

In [ ]:

def top_confusions(predictions_store, p, class_names, top_n=15):
    run = None
    for r in predictions_store:
        if  r["p"] == p:
            run = r
            break

    y_true = run["y_true"]
    y_pred = run["y_pred"]

    wrong_pairs = [(t, pred) for t, pred in zip(y_true, y_pred) if t != pred]
    counts = Counter(wrong_pairs).most_common(top_n)

    
    for (true_idx, pred_idx), count in counts:
        print(
            f"True: {class_names[int(true_idx)].split('-')[1]:30s} "
            f"Pred: {class_names[int(pred_idx)].split('-')[1]:30s} "
            f"Count: {count}"
        )

In [ ]:
top_confusions(predictions_store,  1, class_names, top_n=10)

## Print

In [ ]:
# %load_ext tensorboard
# %tensorboard --logdir logs

In [ ]:
import matplotlib.pyplot as plt

for images, labels in train_variant.take(1):
    plt.figure(figsize=(10, 6))
    for i in range(20):
        plt.subplot(4, 5, i + 1)
        plt.imshow(tf.cast(images[i], tf.uint8).numpy())
        plt.axis("off")
    plt.tight_layout()
    plt.show()